# 08 - Results Chapter Inputs
NFL Play-Type Prediction (2016-2023)
Goal: Additional evaluations of the best model (XGBoost, maxi) on the test set (2022-2023) for the results chapter.

- A) Dataset basics
- B) Confusion matrix
- C) Grouped permutation importance
- D) Error analysis by game situation

No model is persisted by notebook 03, so XGBoost maxi is retrained here with the identical pipeline.
This notebook does **not** write to `metrics.csv`.

## 0. Setup

In [ ]:
import sys
sys.path.append('..')

import io
import contextlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
from sklearn.metrics import roc_auc_score, confusion_matrix

'''Self modules'''
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.models import split_data, get_X_y
from src.xgboost import train_xgboost
from src.analysis import feature_groups, grouped_permutation_importance, bucket_report
from config import FEATURE_SETS, FEATURE_CONFIG, FIGURES_MODELS, TABLES, PLOT_DPI, PLOT_PALETTE

os.makedirs(FIGURES_MODELS, exist_ok=True)
os.makedirs(TABLES, exist_ok=True)

sns.set_theme(style='whitegrid', palette=PLOT_PALETTE)

plt.rcParams['figure.dpi'] = PLOT_DPI
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

# same colour map / alpha convention as 07_results
MODEL_ORDER = ['LogisticRegression', 'XGBoost', 'FNN', 'ResFNN', 'TabNet']
FS_ORDER    = ['mini', 'comprehensive', 'maxi']
PALETTE     = dict(zip(MODEL_ORDER, sns.color_palette('hls', len(MODEL_ORDER))))
FS_ALPHA    = dict(zip(FS_ORDER, [0.35, 0.6, 1]))

MODEL_NAME   = 'XGBoost'
FEATURE_SET  = 'maxi'
TEST_SEASONS = [2022, 2023]
SEED         = 420   # random_state default of src.xgboost.train_xgboost (config.py has no XGBoost seed)
THRESHOLD    = 0.5
MIN_N        = 500   # buckets below this size are flagged as low_n
EXPECTED_AUC = 0.805

pd.set_option('display.float_format', '{:.3f}'.format)
print('Setup complete.')

## 1. Load, Split & Retrain

Identical to `03_classical_models.ipynb`: split on raw data, preprocess the maxi set separately for train and test, align test columns on train columns.

In [ ]:
df_raw = load_data()
df_train_raw, df_test_raw = split_data(df_raw, test_seasons=TEST_SEASONS)

# silence the long preprocessing report (already shown in 03)
with contextlib.redirect_stdout(io.StringIO()):
    df_train = preprocess(df_train_raw, feature_set=FEATURE_SETS[FEATURE_SET], feature_config=FEATURE_CONFIG)
    df_test  = preprocess(df_test_raw,  feature_set=FEATURE_SETS[FEATURE_SET], feature_config=FEATURE_CONFIG)

# align test on train columns (missing = 0, extra = dropped)
df_test = df_test.reindex(columns=df_train.columns, fill_value=0)

X_train, y_train = get_X_y(df_train)
X_test,  y_test  = get_X_y(df_test)
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')

In [ ]:
model = train_xgboost(X_train, y_train, random_state=SEED)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred  = model.predict(X_test)   # threshold 0.5

test_auc = roc_auc_score(y_test, y_proba)
print(f'Test ROC AUC: {test_auc:.4f}  (expected {EXPECTED_AUC:.3f})')
assert round(test_auc, 3) == EXPECTED_AUC, 'ROC AUC deviates from reported value - stop and check'

## 2. A) Dataset Basics

In [ ]:
basics = pd.Series({
    'plays_train':      len(y_train),
    'plays_test':       len(y_test),
    'pass_share_train': round(y_train.mean(), 3),
    'pass_share_test':  round(y_test.mean(), 3),
}, dtype=object)
display(basics.to_frame('value'))

In [ ]:
# feature lists before encoding (+ number of columns after encoding)
encoded_cols = [c for c in df_train.columns if c != 'target']
fs_overview = pd.DataFrame([
    {
        'feature_set':       fs,
        'n_features':        len(cols),
        'n_encoded_columns': sum(any(c == f or c.startswith(f + '_') for f in cols) for c in encoded_cols),
    }
    for fs, cols in FEATURE_SETS.items()
])
display(fs_overview)

for fs, cols in FEATURE_SETS.items():
    print(f'\n{fs} ({len(cols)}):')
    print('  ' + ', '.join(cols))

for f in ['shotgun', 'no_huddle']:
    present = {fs: f in cols for fs, cols in FEATURE_SETS.items()}
    print(f'\n{f} in all sets: {all(present.values())}  {present}')

## 3. B) Confusion Matrix (threshold 0.5)

In [ ]:
labels = ['Run (0)', 'Pass (1)']
cm_abs  = confusion_matrix(y_test, y_pred)
cm_norm = confusion_matrix(y_test, y_pred, normalize='true')

cm_abs_df  = pd.DataFrame(cm_abs,  index=[f'true {l}' for l in labels], columns=[f'pred {l}' for l in labels])
cm_norm_df = pd.DataFrame(cm_norm, index=[f'true {l}' for l in labels], columns=[f'pred {l}' for l in labels])

print('Absolute:')
display(cm_abs_df)
print('Normalised by true class:')
display(cm_norm_df)
print(f'Recall Run : {cm_norm[0, 0]:.3f}')
print(f'Recall Pass: {cm_norm[1, 1]:.3f}')
print(f'Accuracy   : {(y_pred == y_test).mean():.3f}')

## 4. C) Grouped Permutation Importance

Metric: drop in test ROC AUC. One-hot columns of a nominal feature are shuffled jointly (e.g. all `posteam_*` = "posteam"), so team identity appears as a single feature.

In [ ]:
groups = feature_groups(X_test.columns, FEATURE_SETS[FEATURE_SET], FEATURE_CONFIG)
print(f'{len(X_test.columns)} encoded columns -> {len(groups)} groups')

perm = grouped_permutation_importance(model, X_test, y_test, groups, n_repeats=5, random_state=SEED)
print(f"Baseline AUC: {perm.attrs['baseline_auc']:.3f}")

perm_top = perm.head(15).copy()
perm_top['mean ± std'] = perm_top['mean'].map('{:.3f}'.format) + ' ± ' + perm_top['std'].map('{:.3f}'.format)
display(perm_top[['feature', 'n_columns', 'mean', 'std', 'mean ± std']])

In [ ]:
plot_df = perm_top.iloc[::-1]   # largest on top

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(
    plot_df['feature'], plot_df['mean'],
    xerr=plot_df['std'],
    color=PALETTE[MODEL_NAME], alpha=FS_ALPHA[FEATURE_SET],
    edgecolor='white', linewidth=0.6,
    error_kw=dict(ecolor='grey', capsize=3, linewidth=1),
)

ax.set_xlabel('Decrease in ROC-AUC (mean ± std, 5 repeats)')
ax.set_title(f'Permutation Importance - {MODEL_NAME} ({FEATURE_SET}), top 15')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
ax.xaxis.set_major_locator(mticker.MaxNLocator(5))
ax.grid(axis='y', visible=False)

fig.tight_layout()
fig.savefig(os.path.join(FIGURES_MODELS, f'perm_importance_{MODEL_NAME}_{FEATURE_SET}.png'), bbox_inches='tight')
plt.show()

## 5. D) Error Analysis by Game Situation

Per bucket: n, actual pass rate, mean predicted pass probability, calibration gap (predicted − actual, > 0 = model overestimates passing), accuracy, ROC AUC.
Buckets with n < 500 are flagged `low_n` (little evidential value).

Two-minute situation = last 120 s of a half (Q2 or Q4, `half_seconds_remaining` ≤ 120); overtime is counted as "rest".

In [ ]:
qtr_labels = X_test['qtr'].map({1: '1', 2: '2', 3: '3', 4: '4', 5: 'OT'})

BUCKETS = {
    'down': pd.Categorical(X_test['down'].astype(int), categories=[1, 2, 3, 4], ordered=True),
    'ydstogo': pd.cut(X_test['ydstogo'], bins=[0, 3, 6, 10, np.inf],
                      labels=['1-3', '4-6', '7-10', '>10']),
    'quarter': pd.Categorical(qtr_labels, categories=['1', '2', '3', '4', 'OT'], ordered=True),
    'score_differential': pd.cut(X_test['score_differential'],
                                 bins=[-np.inf, -15, -8, -1, 0, 7, 14, np.inf],
                                 labels=['<=-15', '-14 to -8', '-7 to -1', '0', '1 to 7', '8 to 14', '>=15']),
    'two_minute': pd.Categorical(
        np.where(X_test['qtr'].isin([2, 4]) & (X_test['half_seconds_remaining'] <= 120), 'two-minute', 'rest'),
        categories=['two-minute', 'rest'], ordered=True),
}

reports = []
for dim, buckets in BUCKETS.items():
    rep = bucket_report(y_test, y_proba, pd.Series(buckets, index=X_test.index),
                        threshold=THRESHOLD, min_n=MIN_N)
    rep.insert(0, 'dimension', dim)
    reports.append(rep)
    print(f'\n=== {dim} ===')
    display(rep.drop(columns='dimension'))

error_analysis = pd.concat(reports, ignore_index=True)

In [ ]:
# Down x distance: accuracy and n only
dd = pd.DataFrame({
    'down':    BUCKETS['down'],
    'ydstogo': BUCKETS['ydstogo'],
    'correct': (y_pred == y_test.to_numpy()).astype(int),
})
dd_acc = dd.pivot_table(index='down', columns='ydstogo', values='correct', aggfunc='mean', observed=False)
dd_n   = dd.pivot_table(index='down', columns='ydstogo', values='correct', aggfunc='size', observed=False)

print('Accuracy (* = n < 500):')
display(dd_acc.map('{:.3f}'.format) + pd.DataFrame(np.where(dd_n < MIN_N, '*', ''), index=dd_n.index, columns=dd_n.columns))
print('n:')
display(dd_n)

dd_long = pd.concat({'accuracy': dd_acc.stack(), 'n': dd_n.stack()}, axis=1).reset_index()
dd_long['low_n'] = dd_long['n'] < MIN_N

## 6. Save Tables

In [ ]:
# new files only - metrics.csv is not touched
perm.to_csv(TABLES / f'perm_importance_{MODEL_NAME}_{FEATURE_SET}.csv', index=False, float_format='%.3f')
error_analysis.to_csv(TABLES / f'error_analysis_{MODEL_NAME}_{FEATURE_SET}.csv', index=False, float_format='%.3f')
dd_long.to_csv(TABLES / f'error_analysis_down_distance_{MODEL_NAME}_{FEATURE_SET}.csv', index=False, float_format='%.3f')
print(f'Tables saved in {TABLES}')